In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import numpy as np
pd.set_option("display.max_columns", None)

In [2]:
lif_int_res_pla = pd.read_csv('Life_Intake_Research_Placement.csv' )
bio_des_facts =  pd.read_csv('BioProfile_Destructive_Facts.csv')
weather = pd.read_csv('FreemanRanchWeather2015_hourly_cleaned.csv').drop('Unnamed: 0', axis=1)

In [3]:
paths = pd.read_csv('../filename_pmiClass.csv', dtype=str).drop(['year', 'second_year', 'day', 'month'], axis=1)
paths['TXSTDonorNumber'] = (
    paths['image_year'].astype(str) + '.' + paths['sample'].astype(str).str.zfill(3)
)
paths["TXSTDonorNumber"] = paths["TXSTDonorNumber"].astype("string")

print(paths.columns)
paths

Index(['filename', 'sample', 'picture_number', 'image_year', 'date',
       'PMI Class', 'TXSTDonorNumber'],
      dtype='object')


,filename,sample,picture_number,image_year,date,PMI Class,TXSTDonorNumber
0,2011.005.03.15.2011 (7).jpg,005,7,2011,3/15/2011,3,2011.005
1,2011.005.04.20.2011 (1).JPG,005,1,2011,4/20/2011,5,2011.005
2,2011.005.04.04.2011 (7).jpg,005,7,2011,4/4/2011,5,2011.005
3,2011.005.03.30.2011 (1).JPG,005,1,2011,3/30/2011,5,2011.005
4,2011.005.03.21.2011 (2).JPG,005,2,2011,3/21/2011,4,2011.005
...,...,...,...,...,...,...,...
500,2023.003.01.17.2023 (11).JPG,003,11,2023,1/17/2023,2,2023.003
501,2023.003.01.30.2023 (11).JPG,003,11,2023,1/30/2023,4,2023.003
502,2023.003.01.30.2024 (11).JPG,003,11,2023,1/30/2023,5,2023.003
503,2023.003.08.23.2023 (11).JPG,003,11,2023,8/23/2023,5,2023.003


In [4]:
donors = paths['TXSTDonorNumber'].astype(str).unique()
len(donors)

36

In [5]:
drop_cols = [
"In_Temp", "In_Hum",
"In_Dew", "In_Heat", "In_EMC", "In_Air_Density",
"In_Air_ET", "Wind_Samp", "Wind_Tx",   "ISS_Recept", 
"Researchers", "FactsOfDeathID", "CityOfDeath", "StateOfDeath",
"RecordEntryBy", "RecordEntryDate", "RecordAuditBy", "RecordAuditDate",
"RecordUpdateBy", "RecordUpdateDate"]
dfs = {
    "lif_int_res_pla": lif_int_res_pla,
    "bio_des_facts": bio_des_facts,
    "weather": weather,
}

for name, df_ in dfs.items():
    cols_to_drop = [c for c in drop_cols if c in df_.columns]
    df_.drop(columns=cols_to_drop, inplace=True)
    print(name, "dropped:", cols_to_drop)

lif_int_res_pla dropped: ['CityOfDeath', 'StateOfDeath', 'RecordEntryBy', 'RecordEntryDate', 'RecordAuditBy', 'RecordAuditDate', 'RecordUpdateBy', 'RecordUpdateDate']
bio_des_facts dropped: ['Researchers', 'FactsOfDeathID', 'CityOfDeath', 'StateOfDeath', 'RecordEntryBy', 'RecordEntryDate', 'RecordAuditBy', 'RecordAuditDate', 'RecordUpdateBy', 'RecordUpdateDate']
weather dropped: ['In_Temp', 'In_Hum', 'In_Dew', 'In_Heat', 'In_EMC', 'In_Air_Density', 'In_Air_ET', 'Wind_Samp', 'Wind_Tx', 'ISS_Recept']


In [6]:
mask = (weather == '---').all(axis=0)   # True for cols that are all '---'
weather = weather.loc[:, ~mask]
weather

,date,time,Temp_Out,temp_high,temp_low,humidity_out,dew_point,wind_speed,wind_dir,Wind_Run,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate,Heat_D-D,Cool_D-D,Arc_Int
0,2015-01-01,08:00:00,34.6,34.7,34.6,96,33.6,2,N,1.0,6,NNE,33.8,34.6,33.8,30.300,0.01,0.00,0.633,0.0,30
1,2015-01-01,07:00:00,34.7,34.9,34.7,96,33.7,3,NNE,1.5,9,N,32.4,34.7,32.4,30.303,0.02,0.06,0.631,0.0,30
2,2015-01-01,09:00:00,34.7,34.9,34.7,96,33.7,2,N,1.0,7,NNE,33.9,34.7,33.9,30.315,0.00,0.00,0.631,0.0,30
3,2015-01-01,06:00:00,35.2,35.3,34.9,95,33.9,2,NNE,1.0,7,NNE,34.5,35.2,34.5,30.305,0.00,0.00,0.621,0.0,30
4,2015-01-01,05:00:00,35.1,35.4,35.1,95,33.8,2,WNW,1.0,6,NNW,34.4,35.1,34.4,30.318,0.04,0.11,0.623,0.0,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8309,2015-12-31,20:00:00,49.5,49.7,49.5,75,41.9,7,NNE,3.5,14,NE,46.6,49.3,46.4,30.325,0.00,0.00,0.323,0.0,30
8310,2015-12-31,21:00:00,49.2,49.3,49.2,75,41.6,6,NNE,3.0,15,NNE,46.7,48.9,46.4,30.351,0.00,0.00,0.329,0.0,30
8311,2015-12-31,22:00:00,48.9,49.1,48.9,74,41.0,7,NNE,3.5,13,NE,45.9,48.6,45.6,30.344,0.00,0.00,0.335,0.0,30
8312,2015-12-31,23:00:00,48.7,48.9,48.7,75,41.1,8,NNE,4.0,16,N,44.9,48.4,44.6,30.344,0.00,0.00,0.340,0.0,30


In [7]:
weather = weather.copy()

weather["date"] = pd.to_datetime(weather["date"], errors="coerce")

m = weather["date"].dt.month
season_num = (m % 12 + 3) // 3
season_map = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Autumn"}
weather["season"] = season_num.map(season_map)
weather

,date,time,Temp_Out,temp_high,temp_low,humidity_out,dew_point,wind_speed,wind_dir,Wind_Run,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate,Heat_D-D,Cool_D-D,Arc_Int,season
0,2015-01-01,08:00:00,34.6,34.7,34.6,96,33.6,2,N,1.0,6,NNE,33.8,34.6,33.8,30.300,0.01,0.00,0.633,0.0,30,Winter
1,2015-01-01,07:00:00,34.7,34.9,34.7,96,33.7,3,NNE,1.5,9,N,32.4,34.7,32.4,30.303,0.02,0.06,0.631,0.0,30,Winter
2,2015-01-01,09:00:00,34.7,34.9,34.7,96,33.7,2,N,1.0,7,NNE,33.9,34.7,33.9,30.315,0.00,0.00,0.631,0.0,30,Winter
3,2015-01-01,06:00:00,35.2,35.3,34.9,95,33.9,2,NNE,1.0,7,NNE,34.5,35.2,34.5,30.305,0.00,0.00,0.621,0.0,30,Winter
4,2015-01-01,05:00:00,35.1,35.4,35.1,95,33.8,2,WNW,1.0,6,NNW,34.4,35.1,34.4,30.318,0.04,0.11,0.623,0.0,30,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8309,2015-12-31,20:00:00,49.5,49.7,49.5,75,41.9,7,NNE,3.5,14,NE,46.6,49.3,46.4,30.325,0.00,0.00,0.323,0.0,30,Winter
8310,2015-12-31,21:00:00,49.2,49.3,49.2,75,41.6,6,NNE,3.0,15,NNE,46.7,48.9,46.4,30.351,0.00,0.00,0.329,0.0,30,Winter
8311,2015-12-31,22:00:00,48.9,49.1,48.9,74,41.0,7,NNE,3.5,13,NE,45.9,48.6,45.6,30.344,0.00,0.00,0.335,0.0,30,Winter
8312,2015-12-31,23:00:00,48.7,48.9,48.7,75,41.1,8,NNE,4.0,16,N,44.9,48.4,44.6,30.344,0.00,0.00,0.340,0.0,30,Winter


In [8]:
years = [2011, 2012, 2013, 2014, 2015, 2023]

# filter by year
weather_filtered = weather[weather["date"].dt.year.isin(years)]
weather_filtered

,date,time,Temp_Out,temp_high,temp_low,humidity_out,dew_point,wind_speed,wind_dir,Wind_Run,Hi_Speed,Hi_Dir,Wind_Chill,Heat_Index,THW_Index,THSW_Bar,THSW_Rain,rain_rate,Heat_D-D,Cool_D-D,Arc_Int,season
0,2015-01-01,08:00:00,34.6,34.7,34.6,96,33.6,2,N,1.0,6,NNE,33.8,34.6,33.8,30.300,0.01,0.00,0.633,0.0,30,Winter
1,2015-01-01,07:00:00,34.7,34.9,34.7,96,33.7,3,NNE,1.5,9,N,32.4,34.7,32.4,30.303,0.02,0.06,0.631,0.0,30,Winter
2,2015-01-01,09:00:00,34.7,34.9,34.7,96,33.7,2,N,1.0,7,NNE,33.9,34.7,33.9,30.315,0.00,0.00,0.631,0.0,30,Winter
3,2015-01-01,06:00:00,35.2,35.3,34.9,95,33.9,2,NNE,1.0,7,NNE,34.5,35.2,34.5,30.305,0.00,0.00,0.621,0.0,30,Winter
4,2015-01-01,05:00:00,35.1,35.4,35.1,95,33.8,2,WNW,1.0,6,NNW,34.4,35.1,34.4,30.318,0.04,0.11,0.623,0.0,30,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8308,2015-12-31,19:00:00,49.7,49.8,49.7,74,41.7,6,NNE,3.0,10,NNE,47.3,49.4,47.0,30.301,0.00,0.00,0.319,0.0,30,Winter
8309,2015-12-31,20:00:00,49.5,49.7,49.5,75,41.9,7,NNE,3.5,14,NE,46.6,49.3,46.4,30.325,0.00,0.00,0.323,0.0,30,Winter
8310,2015-12-31,21:00:00,49.2,49.3,49.2,75,41.6,6,NNE,3.0,15,NNE,46.7,48.9,46.4,30.351,0.00,0.00,0.329,0.0,30,Winter
8311,2015-12-31,22:00:00,48.9,49.1,48.9,74,41.0,7,NNE,3.5,13,NE,45.9,48.6,45.6,30.344,0.00,0.00,0.335,0.0,30,Winter


In [9]:
weather_filtered.to_csv('FreemanRanchWeather_cleaned.csv', index=False)

In [10]:
missing_pct = lif_int_res_pla.isna().mean()

cols_to_drop = missing_pct[missing_pct > 0.70].index
print(cols_to_drop)
lif_int_res_pla = lif_int_res_pla.drop(columns=cols_to_drop)

Index(['WaistCircumference(cm)', 'DonatedBy', 'Unnamed: 12', 'Unnamed: 13',
       'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16'],
      dtype='object')


In [11]:
missing = bio_des_facts.isna().mean()

cols_to_drop = missing[missing > 0.70].index
print(cols_to_drop)
bio_des_facts = bio_des_facts.drop(columns=cols_to_drop)

Index([], dtype='object')


In [12]:
# Make sure the donor number columns are strings before filter
lif_int_res_pla['TXSTDonorNumber'] = lif_int_res_pla['TXSTDonorNumber'].astype(str)
bio_des_facts['TXSTDonorNumber']   = bio_des_facts['TXSTDonorNumber'].astype(str)

# Filter the DataFrames
lif_int_res_pla_filt = lif_int_res_pla[
    lif_int_res_pla['TXSTDonorNumber'].isin(donors)
]

bio_des_facts_filt = bio_des_facts[
    bio_des_facts['TXSTDonorNumber'].isin(donors)
]

In [13]:
def drop_mostly_missing_rows(df, pct_missing=0.9):
    # keep rows that have at least (1 - pct_missing) non-null values
    thresh = int((1 - pct_missing) * df.shape[1])
    return df.dropna(thresh=thresh)

lif_int_res_pla = drop_mostly_missing_rows(lif_int_res_pla_filt, pct_missing=0.7)
bio_des_facts   = drop_mostly_missing_rows(bio_des_facts_filt,   pct_missing=0.7)
weather         = drop_mostly_missing_rows(weather,         pct_missing=0.7)


In [14]:
print(lif_int_res_pla.shape)
lif_int_res_pla.columns

(45, 57)


Index(['TXSTDonorNumber', 'AgeAtDeath', 'DateOfDeath', 'DODSpecify',
       'ImmediateCOD', 'MannerOfDeath', 'LocationOfDeath', 'FacilityOfDeath',
       'IntakeID', 'DateReceived', 'DateOfIntake', 'InitialPlacementLocation',
       'ApproxTime', 'CadaverStature(cm)', 'FootLength(cm)',
       'CadaverWeight(lbs)', 'PersonalEffects', 'PersonalEffectsDescription',
       'SamplesCollected', 'PhotographID', 'PhotographOverallBody',
       'PhotographAnteriorFace', 'PhotographProfileLeftFace',
       'PhotographProfileRightFace', 'PhotographTeeth', 'PhotographLeftArm',
       'PhotographRightArm', 'PhotographLeftLeg', 'PhotographRightLeg',
       'PhotographScars', 'PhotographTattoos', 'PhotographInjuries',
       'PhotographJewelry', 'Autopsy', 'FullAutopsy', 'AbdominalAutopsy',
       'CranialAutopsy', 'ThoracicAutopsy', 'SpinalAutopsy', 'OrganDonor',
       'DonatedEyes', 'DonatedSkin', 'DonatedBones', 'DonatedInternalOrgans',
       'Paperless', 'Donation', 'TXSTDSC #', 'DOD', 'Receive

In [15]:
lif_int_res_pla

,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,LocationOfDeath,FacilityOfDeath,IntakeID,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),FootLength(cm),CadaverWeight(lbs),PersonalEffects,PersonalEffectsDescription,SamplesCollected,PhotographID,PhotographOverallBody,PhotographAnteriorFace,PhotographProfileLeftFace,PhotographProfileRightFace,PhotographTeeth,PhotographLeftArm,PhotographRightArm,PhotographLeftLeg,PhotographRightLeg,PhotographScars,PhotographTattoos,PhotographInjuries,PhotographJewelry,Autopsy,FullAutopsy,AbdominalAutopsy,CranialAutopsy,ThoracicAutopsy,SpinalAutopsy,OrganDonor,DonatedEyes,DonatedSkin,DonatedBones,DonatedInternalOrgans,Paperless,Donation,TXSTDSC #,DOD,Received by FACTS,Initial Placement,Length,Place of death,State,Donation.1,Curated,Unnamed: 10,Unnamed: 11
36,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated
39,2011.008,53,2011-06-08,NaN,Lung cancer,Natural,Hospital inpatient,Christus Santa Rosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D08-2011,2011.008,2011-06-08 00:00:00,2011-06-09 00:00:00,Surface,1,"New Braunfels, TX",TX,NOK,Y,3/30/2012 buried,10/21/12 excavated
40,2011.008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38.0,2011-06-09,2011-06-09,Surface,1218.0,173.0,23,NaN,%,NaN,%,False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D08-2011,2011.008,2011-06-08 00:00:00,2011-06-09 00:00:00,Surface,1,"New Braunfels, TX",TX,NOK,Y,3/30/2012 buried,10/21/12 excavated
43,2011.011,75,2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN,91.0,2011-07-19,2011-07-19,NaN,NaN,173.0,NaN,NaN,Z,NaN,Z,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,False,False,False,False,False,NaN,False,False,False,False,False,D11-2011,2011.011,2011-07-18 00:00:00,2011-07-19 00:00:00,Surface,1,"Llano, TX",TX,Living,Y,1/2/2012 buried,3/25/2012 excavated
55,2011.023,66,2011-12-18,NaN,End Stage renal disease,Natural,Nursing home,Guadalupe Valley Nursing Center LP,118.0,2011-12-19,2011-12-19,Surface,NaN,163.0,22.5,NaN,q,NaN,q,True,True,True,False,False,True,True,True,True,True,True,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D23-2011,2011.023,2011-12-18 00:00:00,2011-12-19 00:00:00,Surface,1,"Seguin, TX",TX,Living,Y,1/16/2013 Picked up and placed in cage @ FARF,1/17/2013 brought to ORPL
59,2012.002,68,2012-01-06,NaN,PAI Respiratory Failure (Acute),Natural,Hospital inpatient,St. Lukes Patients Medical Center,120.0,2012-01-06,2012-01-06,Surface,NaN,154.0,23,NaN,s,NaN,s,True,True,True,False,False,True,True,True,True,True,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D02-2012,2012.002,2012-01-06 00:00:00,2012-01-06 00:00:00,Surface,0,"Pasadena, TX",TX,NOK,Y,2/1/13 picked up and in bin @farf,5/19/13 buried
60,2012.003,78,2012-01-27,NaN,Metastatic Lung Cancer,Natural,Hospital inpatient,"Llano Memorial Healthcare System, Memorial Hos...",121.0,2012-01-28,2012-01-28,Surface,NaN,158.0,21,NaN,t,"hospital gown, bible",t,True,True,True,False,False,True,True,True,True,True,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D03-2012,2012.003,2012-01-27 00:00:00,2012-01-28 00:00:00,Surface,1,"Llano, TX",TX,Living,Y,6/15/12 buried,4/2014 Curated
61,2012.004,63,2012-01-27,NaN,Metastatic Breast Cancer,Natural,Hospital inpatient,Scott and White Memorial Hospital,122.0,201

In [16]:
print(bio_des_facts.shape)
bio_des_facts.columns

(51, 23)


Index(['TXSTDonorNumber', 'Curated', 'Current Facility', 'AgeAtDeath', 'Sex',
       'Race', 'HispanicOrigin', 'Height(cm)', 'HeightEstimated',
       'CadaverStature(cm)', 'Weight(lb)', 'WeightEstimated',
       'CadaverWeight(lbs)', 'Destructive Sampling', 'SamplingType',
       'ElementsSampled', 'SpecifySamples', 'DateOfDeath', 'DODSpecify',
       'ImmediateCOD', 'MannerOfDeath', 'LocationOfDeath', 'FacilityOfDeath'],
      dtype='object')

In [17]:
bio_des_facts

,TXSTDonorNumber,Curated,Current Facility,AgeAtDeath,Sex,Race,HispanicOrigin,Height(cm),HeightEstimated,CadaverStature(cm),Weight(lb),WeightEstimated,CadaverWeight(lbs),Destructive Sampling,SamplingType,ElementsSampled,SpecifySamples,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,LocationOfDeath,FacilityOfDeath
63,2011.005,True,GEFARL,80.0,Male,White,Non-hispanic,178.0,True,178.0,160.0,True,NaN,True,6831,6831,Left 6th rib midshaft,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN
73,2011.011,True,GEFARL,75.0,Male,White,NaN,NaN,False,173.0,218.0,False,NaN,True,91,91,"Occipital, mandible, left and right: 6th rib, ...",2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN
74,2011.011,True,GEFARL,75.0,Male,White,NaN,NaN,False,173.0,218.0,False,NaN,True,6834,6834,Left 5th rib midshaft,2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN
75,2011.011,True,GEFARL,75.0,Male,White,NaN,NaN,False,173.0,218.0,False,NaN,True,9254,9254,2.5 cm cortical bone fragment taken from the d...,2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN
89,2012.002,True,GEFARL,68.0,Female,White,Non-hispanic,157.0,False,154.0,130.0,True,NaN,True,6855,6855,Left 6th rib midshaft,2012-01-06,NaN,PAI Respiratory Failure (Acute),Natural,Hospital inpatient,St. Lukes Patients Medical Center
90,2012.003,True,GEFARL,78.0,Female,White,Non-hispanic,163.0,True,158.0,125.0,True,NaN,True,92,92,"Teeth #3 and #27, occipital, mandible, left an...",2012-01-27,NaN,Metastatic Lung Cancer,Natural,Hospital inpatient,"Llano Memorial Healthcare System, Memorial Hos..."
91,2012.003,True,GEFARL,78.0,Female,White,Non-hispanic,163.0,True,158.0,125.0,True,NaN,True,248,248,"Tooth #4, left rib 7",2012-01-27,NaN,Metastatic Lung Cancer,Natural,Hospital inpatient,"Llano Memorial Healthcare System, Memorial Hos..."
92,2012.003,True,GEFARL,78.0,Female,White,Non-hispanic,163.0,True,158.0,125.0,True,NaN,True,9273,9273,2.5 cm cortical bone fragment taken from the d...,2012-01-27,NaN,Metastatic Lung Cancer,Natural,Hospital inpatient,"Llano Memorial Healthcare System, Memorial Hos..."
93,2012.004,True,GEFARL,63.0,Female,White,Non-hispanic,168.0,False,160.0,154.0,False,NaN,True,93,93,"Teeth #5 and #20, occipital, mandible, left an...",2012-01-27,NaN,Metastatic Breast Cancer,Natural,Hospital inpatient,Scott and White Memorial Hospital
94,2012.004,True,GEFARL,63.0,Female,White,Non-hispanic,168.0,False,160.0,154.0,False,NaN,True,221,221,Tooth #26,2012-01-27,NaN,Metastatic Breast Cancer,Natural,Hospital inpatient,Scott and White Memorial Hospital


In [18]:
def clean_age(col):
    # 1) Strip spaces, normalize actual NaNs and string 'nan'
    col = col.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "NaN": np.nan})
    # 2) Coerce to numeric, impossible values become NaN
    col = pd.to_numeric(col, errors="coerce")
    # 3a) If you want true missing values:
    return col.astype("Int64")

    # 3b) OR, if you want to fill with 189:
    # return col.fillna(189).astype(int)

lif_int_res_pla["AgeAtDeath"] = clean_age(lif_int_res_pla["AgeAtDeath"])
bio_des_facts["AgeAtDeath"]   = clean_age(bio_des_facts["AgeAtDeath"])

["TXSTDonorNumber", "AgeAtDeath",  "CadaverStature(cm)", "CadaverWeight(lbs)", "DateOfDeath", "DODSpecify",
         "ImmediateCOD", "MannerOfDeath", "LocationOfDeath", "FacilityOfDeath"]

In [19]:
# 1) Merge lif + bio on donor
lif_bio = lif_int_res_pla.merge(
    bio_des_facts,
    on=["TXSTDonorNumber", "AgeAtDeath",  "CadaverStature(cm)", "CadaverWeight(lbs)", "DateOfDeath", "DODSpecify",
         "ImmediateCOD", "MannerOfDeath", "LocationOfDeath", "FacilityOfDeath"],
    how="outer",          # change to 'inner' if you only want common donors
    #suffixes=("_lif", "_bio")
)
lif_bio["TXSTDonorNumber"] = lif_bio["TXSTDonorNumber"].astype("string")
lif_bio

,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,LocationOfDeath,FacilityOfDeath,IntakeID,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),FootLength(cm),CadaverWeight(lbs),PersonalEffects,PersonalEffectsDescription,SamplesCollected,PhotographID,PhotographOverallBody,PhotographAnteriorFace,PhotographProfileLeftFace,PhotographProfileRightFace,PhotographTeeth,PhotographLeftArm,PhotographRightArm,PhotographLeftLeg,PhotographRightLeg,PhotographScars,PhotographTattoos,PhotographInjuries,PhotographJewelry,Autopsy,FullAutopsy,AbdominalAutopsy,CranialAutopsy,ThoracicAutopsy,SpinalAutopsy,OrganDonor,DonatedEyes,DonatedSkin,DonatedBones,DonatedInternalOrgans,Paperless,Donation,TXSTDSC #,DOD,Received by FACTS,Initial Placement,Length,Place of death,State,Donation.1,Curated_x,Unnamed: 10,Unnamed: 11,Curated_y,Current Facility,Sex,Race,HispanicOrigin,Height(cm),HeightEstimated,Weight(lb),WeightEstimated,Destructive Sampling,SamplingType,ElementsSampled,SpecifySamples
0,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
1,2011.008,53,2011-06-08,NaN,Lung cancer,Natural,Hospital inpatient,Christus Santa Rosa,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D08-2011,2011.008,2011-06-08 00:00:00,2011-06-09 00:00:00,Surface,1,"New Braunfels, TX",TX,NOK,Y,3/30/2012 buried,10/21/12 excavated,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011.008,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,38.0,2011-06-09,2011-06-09,Surface,1218.0,173.0,23,NaN,%,NaN,%,False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D08-2011,2011.008,2011-06-08 00:00:00,2011-06-09 00:00:00,Surface,1,"New Braunfels, TX",TX,NOK,Y,3/30/2012 buried,10/21/12 excavated,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011.011,75,2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN,91.0,2011-07-19,2011-07-19,NaN,NaN,173.0,NaN,NaN,Z,NaN,Z,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,False,False,False,False,False,NaN,False,False,False,False,False,D11-2011,2011.011,2011-07-18 00:00:00,2011-07-19 00:00:00,Surface,1,"Llano, TX",TX,Living,Y,1/2/2012 buried,3/25/2012 excavated,True,GEFARL,Male,White,NaN,NaN,False,218.0,False,True,91.0,91.0,"Occipital, mandible, left and right: 6th rib, ..."
4,2011.011,75,2011-07-18,NaN,Metastatic Lung Cancer,Natural,Private residence,NaN,91.0,2011-07-19,2011-07-19,NaN,NaN,173.0,NaN,NaN,Z,NaN,Z,False,False,False,False,False,False,False,False,False,False,False,False,False,False,NaN,False,False,False,False,False,NaN,False,False,False,False,False,D11-2011,2011.011,2011-07-18 00:00:00,2011-07-19 00:00:00,Surface,1,"Llano, TX",TX,Living,Y,1/2/2012 buried,3/25/2012 excavated,True,GEFARL,Male,White,NaN,NaN,False,218.0,False,True,6834.0,6834.0,Left 5th rib midshaft
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,2015.009,74,2015-02-25,Actual,cardiac arrest,Natural,Hospital inpatient,Heart Hospital of Austin,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D09-2015,2015.009,

In [20]:
donors = lif_bio['TXSTDonorNumber'].astype(str).unique()
len(donors)

34

In [21]:
lif_bio.columns

Index(['TXSTDonorNumber', 'AgeAtDeath', 'DateOfDeath', 'DODSpecify',
       'ImmediateCOD', 'MannerOfDeath', 'LocationOfDeath', 'FacilityOfDeath',
       'IntakeID', 'DateReceived', 'DateOfIntake', 'InitialPlacementLocation',
       'ApproxTime', 'CadaverStature(cm)', 'FootLength(cm)',
       'CadaverWeight(lbs)', 'PersonalEffects', 'PersonalEffectsDescription',
       'SamplesCollected', 'PhotographID', 'PhotographOverallBody',
       'PhotographAnteriorFace', 'PhotographProfileLeftFace',
       'PhotographProfileRightFace', 'PhotographTeeth', 'PhotographLeftArm',
       'PhotographRightArm', 'PhotographLeftLeg', 'PhotographRightLeg',
       'PhotographScars', 'PhotographTattoos', 'PhotographInjuries',
       'PhotographJewelry', 'Autopsy', 'FullAutopsy', 'AbdominalAutopsy',
       'CranialAutopsy', 'ThoracicAutopsy', 'SpinalAutopsy', 'OrganDonor',
       'DonatedEyes', 'DonatedSkin', 'DonatedBones', 'DonatedInternalOrgans',
       'Paperless', 'Donation', 'TXSTDSC #', 'DOD', 'Receive

In [22]:
# 2) Add paths (filenames, images) on donor
lif_bio_paths = paths.merge(
    lif_bio,
    on="TXSTDonorNumber",
    how="inner"            # one donor can have many path rows
)
lif_bio_paths

,filename,sample,picture_number,image_year,date,PMI Class,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,LocationOfDeath,FacilityOfDeath,IntakeID,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),FootLength(cm),CadaverWeight(lbs),PersonalEffects,PersonalEffectsDescription,SamplesCollected,PhotographID,PhotographOverallBody,PhotographAnteriorFace,PhotographProfileLeftFace,PhotographProfileRightFace,PhotographTeeth,PhotographLeftArm,PhotographRightArm,PhotographLeftLeg,PhotographRightLeg,PhotographScars,PhotographTattoos,PhotographInjuries,PhotographJewelry,Autopsy,FullAutopsy,AbdominalAutopsy,CranialAutopsy,ThoracicAutopsy,SpinalAutopsy,OrganDonor,DonatedEyes,DonatedSkin,DonatedBones,DonatedInternalOrgans,Paperless,Donation,TXSTDSC #,DOD,Received by FACTS,Initial Placement,Length,Place of death,State,Donation.1,Curated_x,Unnamed: 10,Unnamed: 11,Curated_y,Current Facility,Sex,Race,HispanicOrigin,Height(cm),HeightEstimated,Weight(lb),WeightEstimated,Destructive Sampling,SamplingType,ElementsSampled,SpecifySamples
0,2011.005.03.15.2011 (7).jpg,005,7,2011,3/15/2011,3,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
1,2011.005.04.20.2011 (1).JPG,005,1,2011,4/20/2011,5,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
2,2011.005.04.04.2011 (7).jpg,005,7,2011,4/4/2011,5,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
3,2011.005.03.30.2011 (1).JPG,005,1,2011,3/30/2011,5,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
4,2011.005.03.21.2011 (2).JPG,005,2,2011,3/21/2011,4,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [23]:
# 2) Add paths (filenames, images) on donor
lif_bio_paths = lif_bio.merge(
    paths,
    on="TXSTDonorNumber",
    how="outer"            # one donor can have many path rows
)
lif_bio_paths

,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,LocationOfDeath,FacilityOfDeath,IntakeID,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),FootLength(cm),CadaverWeight(lbs),PersonalEffects,PersonalEffectsDescription,SamplesCollected,PhotographID,PhotographOverallBody,PhotographAnteriorFace,PhotographProfileLeftFace,PhotographProfileRightFace,PhotographTeeth,PhotographLeftArm,PhotographRightArm,PhotographLeftLeg,PhotographRightLeg,PhotographScars,PhotographTattoos,PhotographInjuries,PhotographJewelry,Autopsy,FullAutopsy,AbdominalAutopsy,CranialAutopsy,ThoracicAutopsy,SpinalAutopsy,OrganDonor,DonatedEyes,DonatedSkin,DonatedBones,DonatedInternalOrgans,Paperless,Donation,TXSTDSC #,DOD,Received by FACTS,Initial Placement,Length,Place of death,State,Donation.1,Curated_x,Unnamed: 10,Unnamed: 11,Curated_y,Current Facility,Sex,Race,HispanicOrigin,Height(cm),HeightEstimated,Weight(lb),WeightEstimated,Destructive Sampling,SamplingType,ElementsSampled,SpecifySamples,filename,sample,picture_number,image_year,date,PMI Class
0,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft,2011.005.03.15.2011 (7).jpg,005,7,2011,3/15/2011,3
1,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft,2011.005.04.20.2011 (1).JPG,005,1,2011,4/20/2011,5
2,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft,2011.005.04.04.2011 (7).jpg,005,7,2011,4/4/2011,5
3,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft,2011.005.03.30.2011 (1).JPG,005,1,2011,3/30/2011,5
4,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,NaN,NaN,34.0,2011-03-07,2011-03-07,Burial,1332.0,178.0,23,NaN,"""",NaN,"""",False,False,False,False,False,False,False,False,False,False,False,False,False,False,No,False,False,False,False,False,No,False,False,False,False,False,D05-2011,2011.005,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,"New Braunfels, TX",TX,Living,Y,5/4/2011 buried,12/14/11 excavated,True,GEFARL,Male,White,Non-hispanic,178.0,True,160.0,True,True,6831.0,6831.0,Left 6th rib midshaft,2011.005.03.21.2011 (2).JPG,005,2,2011,3/21/2011,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [24]:
cols = ["LocationOfDeath", "FacilityOfDeath", "IntakeID", "FootLength(cm)", "CadaverWeight(lbs)", "PersonalEffects", "PersonalEffectsDescription", "SamplesCollected",
"PhotographAnteriorFace", "PhotographProfileLeftFace", "PhotographProfileRightFace", "PhotographTeeth", "PhotographLeftArm", "PhotographRightArm",
"PhotographLeftLeg", "PhotographRightLeg", "PhotographScars", "PhotographTattoos", "PhotographInjuries", "PhotographJewelry", "AbdominalAutopsy",
"CranialAutopsy", "ThoracicAutopsy", "SpinalAutopsy", "DonatedEyes", "DonatedBones", "DonatedInternalOrgans", "Paperless", "Donation", "TXSTDSC #",
"Place of death", "State", "Donation.1", "Curated_x", "Unnamed: 10", "Unnamed: 11", "Curated_y", "Current Facility", "HeightEstimated", "WeightEstimated",
"Destructive Sampling", "SamplingType", "ElementsSampled", "SpecifySamples"]
lif_bio_paths = lif_bio_paths.drop(cols, axis=1)
lif_bio_paths

,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),PhotographID,PhotographOverallBody,Autopsy,FullAutopsy,OrganDonor,DonatedSkin,DOD,Received by FACTS,Initial Placement,Length,Sex,Race,HispanicOrigin,Height(cm),Weight(lb),filename,sample,picture_number,image_year,date,PMI Class
0,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,False,False,No,False,No,False,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.15.2011 (7).jpg,005,7,2011,3/15/2011,3
1,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,False,False,No,False,No,False,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.04.20.2011 (1).JPG,005,1,2011,4/20/2011,5
2,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,False,False,No,False,No,False,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.04.04.2011 (7).jpg,005,7,2011,4/4/2011,5
3,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,False,False,No,False,No,False,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.30.2011 (1).JPG,005,1,2011,3/30/2011,5
4,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,False,False,No,False,No,False,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.21.2011 (2).JPG,005,2,2011,3/21/2011,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1260,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,True,True,No,False,No,False,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.01.17.2023 (11).JPG,003,11,2023,1/17/2023,2
1261,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,True,True,No,False,No,False,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.01.30.2023 (11).JPG,003,11,2023,1/30/2023,4
1262,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,True,True,No,False,No,False,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.01.30.2024 (11).JPG,003,11,2023,1/30/2023,5
1263,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,True,True,No,False,No,False,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.08.23.2023 (11).JPG,003,11,2023,8/23/2023,5


In [26]:
lif_bio_paths = lif_bio_paths.sort_values(["TXSTDonorNumber", "filename"])

lif_bio_paths = (
    lif_bio_paths
    .groupby(["TXSTDonorNumber", "filename"], group_keys=False)
    .apply(lambda g: g.ffill().bfill())
    .infer_objects(copy=False)   # fix the fillna/ffill/bfill downcasting warning
)


C:\Users\mirna\AppData\Local\Temp\ipykernel_14392\1146325411.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .apply(lambda g: g.ffill().bfill())
C:\Users\mirna\AppData\Local\Temp\ipykernel_14392\1146325411.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.ffill().bfill())


In [27]:
# If Race is "White" → HispanicOrigin = "Non-hispanic"
lif_bio_paths.loc[lif_bio_paths["Race"] == "White", "HispanicOrigin"] = "Non-hispanic"

# If Race is "Hispanic" → HispanicOrigin = "Hispanic"
lif_bio_paths.loc[lif_bio_paths["Race"] == "Hispanic", "HispanicOrigin"] = "Hispanic"
lif_bio_paths

,TXSTDonorNumber,AgeAtDeath,DateOfDeath,DODSpecify,ImmediateCOD,MannerOfDeath,DateReceived,DateOfIntake,InitialPlacementLocation,ApproxTime,CadaverStature(cm),PhotographID,PhotographOverallBody,Autopsy,FullAutopsy,OrganDonor,DonatedSkin,DOD,Received by FACTS,Initial Placement,Length,Sex,Race,HispanicOrigin,Height(cm),Weight(lb),filename,sample,picture_number,image_year,date,PMI Class
6,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,0.0,0.0,No,0.0,No,0.0,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.07.2011 (15).JPG,005,15,2011,3/7/2011,1
10,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,0.0,0.0,No,0.0,No,0.0,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.11.2011 (9).JPG,005,9,2011,3/11/2011,2
0,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,0.0,0.0,No,0.0,No,0.0,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.15.2011 (7).jpg,005,7,2011,3/15/2011,3
4,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,0.0,0.0,No,0.0,No,0.0,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.21.2011 (2).JPG,005,2,2011,3/21/2011,4
12,2011.005,80,2011-03-07,NaN,Alzheimer's disease,Natural,2011-03-07,2011-03-07,Burial,1332.0,178.0,0.0,0.0,No,0.0,No,0.0,2011-03-07 00:00:00,2011-03-07 00:00:00,Surface,0,Male,White,Non-hispanic,178.0,160.0,2011.005.03.23.2011 (1).JPG,005,1,2011,3/23/2011,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1263,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,1.0,1.0,No,0.0,No,0.0,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.08.23.2023 (11).JPG,003,11,2023,8/23/2023,5
1252,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,1.0,1.0,No,0.0,No,0.0,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.09.13.2023 (11).JPG,003,11,2023,9/13/2023,5
1255,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,1.0,1.0,No,0.0,No,0.0,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.10.18.2023 (12).JPG,003,12,2023,10/18/2023,5
1245,2023.003,35,2023-01-05,Actual,Acute myelocytic leukemia (AML),Natural,2023-01-09,2023-01-09,Cooler,1645.0,166.0,1.0,1.0,No,0.0,No,0.0,2023-01-05 00:00:00,2023-01-09 00:00:00,cooler @ 1645,4,NaN,NaN,NaN,NaN,NaN,2023.003.11.15.2023 (12).JPG,003,12,2023,11/15/2023,5


In [28]:
lif_bio_paths = lif_bio_paths[
    ~lif_bio_paths["TXSTDonorNumber"].isin([2014.02, 2015.03])
]


In [29]:
lif_bio_paths['filename'].nunique()

504

In [30]:
donors = lif_bio_paths['TXSTDonorNumber'].astype(str).unique()
len(donors)

36

In [31]:
lif_bio_paths.to_excel('donorMetadata.xlsx', index=False)